In [1]:
import pathlib

import bioio_lif
import matplotlib.pyplot as plt
import numpy as np
from PIL import Image
from bioio import BioImage

In [2]:
parent_target_dir = pathlib.Path("./output/")


# LIF Support
**Fully Functional**

In [3]:
lif_path = "/Users/erik/Documents/Promotion/Projekte/Anjas_Stuff/_data/Segmentation Training Data/28-06-2024/HEK293_CellMaskDR_LessDense_01.lif"

lif_target_dir = parent_target_dir / "lif"
target_dir = lif_target_dir
target_dir.mkdir(parents=True, exist_ok=True)

bio_image = BioImage(lif_path, reader=bioio_lif.Reader)  # Specify the backend explicitly
data = np.squeeze(bio_image.data)
is_3d = (data.ndim >= 4 and data.shape[1] > 1)
# ToDo EK Long Term: Unify 2D and 3D loading to prevent double loading in 3D Case


# get all series in the lif file
scenes = bio_image.scenes
total_scenes = len(scenes)

for index, scene_id in enumerate(scenes):
    scene = scene_id

    #remove the unnecessary data in the array
    bio_image.set_scene(scene)
    #TCZXY 5D array
    npy_array = bio_image.data

    squeezed_img = np.squeeze(npy_array)
    n_channels = squeezed_img.shape[0]

    for channel_id in range(n_channels):
        # Extract the height and width of the image
        image = squeezed_img[channel_id]
        img = Image.fromarray(image)

        file_name = f"{scene}{"_c"}{channel_id + 1}.tif"
        target_path = target_dir / file_name
        img.save(str(target_path))

# CZI Support

In [4]:
# !pip install bioio bioio-czi

In [5]:
from bioio import BioImage
import bioio_czi

In [ ]:
czi_path = "/Users/erik/Downloads/CellSePi Segmentation Data/Zeiss_Test_Images/20X_SR-Airyscan_JDCV.czi"
czi_path = "/Users/erik/Downloads/CellSePi Segmentation Data/Zeiss_Test_Images/Tumor_HE_Orig_small.czi"

czi_target_dir = parent_target_dir / "czi"
target_dir = czi_target_dir
target_dir.mkdir(parents=True, exist_ok=True)

# Load the file
czi_img = BioImage(czi_path, reader=bioio_czi.Reader)

# Check dimensions (e.g., T, C, Z, Y, X)
dims = czi_img.dims
print(f"Dimensions: {czi_img.dims}")
print(f"Shape: {czi_img.shape}")

# Access image data as a numpy-backed array (TCZYX)
data = czi_img.data

# Or as an xarray for labeled dimensions
xdata = czi_img.xarray_data

scenes = czi_img.scenes
total_scenes = len(scenes)

for index, scene_id in enumerate(scenes):
    scene = scene_id

    #remove the unnecessary data in the array
    czi_img.set_scene(scene)
    #TCZXY 5D array
    npy_array = czi_img.data

    # squeezed_img = np.squeeze(npy_array)
    # n_channels = dims.C
    n_x = dims.X
    n_y = dims.Y
    n_z = dims.Z

    # Convert to xarray
    xdata = czi_img.xarray_data  # Assuming S is already in the coordinates

    # Stack S into C
    if "S" in czi_img.dims.order:
        xdata = xdata.stack(new_C=("C", "S")).transpose("T", "new_C", "Z", "Y", "X")
    npy_array = xdata.values
    n_channels = xdata.dims[1]

    fig, axes = plt.subplots(ncols=n_channels)
    for channel_id in range(n_channels):
        # Extract the height and width of the image
        image = squeezed_img[channel_id]
        axes[channel_id].imshow(npy_array[0, channel_id, 0].transpose())

        if False:
            img = Image.fromarray(image)

            file_name = f"{scene}{"_c"}{channel_id + 1}.tif"
            target_path = target_dir / file_name
            img.save(str(target_path))

In [ ]:
estimated_bytes = czi_img.data.nbytes
print(f"Required RAM: {estimated_bytes / (1024 ** 2):.2f} MB")

In [ ]:
czi_img.channel_names

In [ ]:
czi_img.dims.order

In [ ]:
from bioio_base.transforms import reshape_data
from bioio import BioImage, Dimensions


class MergedSImage:
    def __init__(self, file_path, reader=None):
        self._img = BioImage(file_path, reader=reader)
        print(
            f"Loaded image with dimensions: {self._img.dims.order} ({self._img.dims.C} channels, {self._img.dims.S} slices)"
        )
        if "M" in self._img.dims.order:
            print(f"Warning: Image has M dimension, which is not supported.")
            raise Exception("Image has M dimension, which is not supported. (Mosaic or tiling images)")
        self.has_s = "S" in self._img.dims.order and self._img.dims.S > 1

    @property
    def dims(self):
        """Returns a fresh Dimensions object reflecting the merged C and S."""
        d = self._img.dims
        if self.has_s:
            # Create a new Dimensions object from scratch.
            # This ensures the __repr__ (print output) is correctly recalculated.
            d = Dimensions(dims=["T", "C", "Z", "Y", "X"], shape=(d.T, d.C * d.S, d.Z, d.Y, d.X))
        return d

    # @property
    # def shape(self):
    #     """Returns the shape tuple matching the new dimensions (TCZYX)."""
    #     d = self.dims
    #     return (d.T, d.C, d.Z, d.Y, d.X)

    @property
    def data(self):
        """Returns the 5D TCZYX array with S merged into C."""
        return self.get_image_data("TCZYX")

    def get_image_data(self, out_dims="TCZYX", **kwargs):
        """Retrieves data and automatically handles S-to-C merging."""
        if not self.has_s:
            return self._img.get_image_data(out_dims, **kwargs)

        # Force retrieve with S to perform the merge
        raw = self._img.get_image_data("TCZYXS", **kwargs)

        # Reshape: (T, C, Z, Y, X, S) -> (T, C, S, Z, Y, X) -> (T, C*S, Z, Y, X)
        # We move S next to C then flatten them
        t, c, z, y, x, s = raw.shape
        merged = raw.transpose(0, 1, 5, 2, 3, 4).reshape(t, c * s, z, y, x)

        # If the user requested a specific subset of dimensions (like "CYX"),
        # we'd need more logic, but for standard TCZYX this is the core:
        return reshape_data(
            data=merged,
            given_dims="TCZYX",
            return_dims=out_dims
        )

    # Delegate other common attributes to the internal BioImage object
    def __getattr__(self, name):
        return getattr(self._img, name)



In [7]:
 # Usage
wrapped_img = MergedSImage(czi_path)
print(f"Original C: {wrapped_img._img.dims.C}, S: {wrapped_img._img.dims.S}")
print(f"New Dimensions: {wrapped_img.dims}")
full_data = wrapped_img.data  # Now (T, C_new, Z, Y, X)

NameError: name 'MergedSImage' is not defined

In [8]:
wrapped_img.dims

NameError: name 'wrapped_img' is not defined

In [9]:
wrapped_img.shape

NameError: name 'wrapped_img' is not defined

In [10]:
from backend.main_window.data_util import FileType

for elem in FileType:
    print(elem.name)

ImportError: cannot import name 'FileType' from 'backend.main_window.data_util' (/Users/erik/Documents/Promotion/PythonProjects/CellSePi/src/cellsepi/backend/main_window/data_util.py)

In [11]:
from cellsepi.backend.main_window.data_util import FileType

np.where([elem == list(FileType)[2] for elem in FileType])[0].item()

ImportError: cannot import name 'FileType' from 'cellsepi.backend.main_window.data_util' (/Users/erik/Documents/Promotion/PythonProjects/CellSePi/src/cellsepi/backend/main_window/data_util.py)

In [17]:
from backend.main_window.constants import SourceType, FileType, DirectoryManager



['lif', 'nd2', 'czi']

In [18]:
pathlib.Path.home()

PosixPath('/Users/erik')

In [5]:
from backend.main_window.constants import FileType, SourceType, DirectoryManager
DirectoryManager().cache_directory

PosixPath('/Users/erik/.cellsepi/intermediate')